# Lab Assignment 1

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA = '/content/drive/MyDrive/Images/'
except ImportError:
    DATA = ''

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import MultinomialNB, GaussianNB
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics import (accuracy_score, confusion_matrix, classification_report,
                             precision_score, recall_score, f1_score)

Mounted at /content/drive


In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import make_pipeline

voice = pd.read_csv(DATA + 'voice.csv')
print(voice.shape)
print(voice['label'].value_counts())
voice.head()

(3168, 21)
label
male      1584
female    1584
Name: count, dtype: int64


,meanfreq,sd,median,Q25,Q75,IQR,skew,kurt,sp.ent,sfm,...,centroid,meanfun,minfun,maxfun,meandom,mindom,maxdom,dfrange,modindx,label
0,0.059781,0.064241,0.032027,0.015071,0.090193,0.075122,12.863462,274.402906,0.893369,0.491918,...,0.059781,0.084279,0.015702,0.275862,0.007812,0.007812,0.007812,0.000000,0.000000,male
1,0.066009,0.067310,0.040229,0.019414,0.092666,0.073252,22.423285,634.613855,0.892193,0.513724,...,0.066009,0.107937,0.015826,0.250000,0.009014,0.007812,0.054688,0.046875,0.052632,male
2,0.077316,0.083829,0.036718,0.008701,0.131908,0.123207,30.757155,1024.927705,0.846389,0.478905,...,0.077316,0.098706,0.015656,0.271186,0.007990,0.007812,0.015625,0.007812,0.046512,male
3,0.151228,0.072111,0.158011,0.096582,0.207955,0.111374,1.232831,4.177296,0.963322,0.727232,...,0.151228,0.088965,0.017798,0.250000,0.201497,0.007812,0.562500,0.554688,0.247119,male
4,0.135120,0.079146,0.124656,0.078720,0.206045,0.127325,1.101174,4.333713,0.971955,0.783568,...,0.135120,0.106398,0.016931,0.266667,0.712812,0.007812,5.484375,5.476562,0.208274,male


In [ ]:
Xv = voice.drop(columns='label')
yv = voice['label'].map({'male': 1, 'female': 0})
Xv_train, Xv_test, yv_train, yv_test = train_test_split(Xv, yv, test_size=0.3, random_state=42, stratify=yv)

base = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5)).fit(Xv_train, yv_train)
yv_pred = base.predict(Xv_test)
print("Baseline accuracy (all features):", accuracy_score(yv_test, yv_pred))
print(classification_report(yv_test, yv_pred, target_names=['female', 'male']))

Baseline accuracy (all features): 0.9737118822292324
              precision    recall  f1-score   support

      female       0.98      0.97      0.97       476
        male       0.97      0.98      0.97       475

    accuracy                           0.97       951
   macro avg       0.97      0.97      0.97       951
weighted avg       0.97      0.97      0.97       951



In [ ]:
from sklearn.feature_selection import SequentialFeatureSelector

feature_names = list(Xv.columns)
scores = {}
for n in range(1, len(feature_names) + 1):
    sfs = SequentialFeatureSelector(
        make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5)),
        n_features_to_select=n, direction='forward', cv=5, n_jobs=-1)
    sfs.fit(Xv_train, yv_train)
    cols = [f for f, s in zip(feature_names, sfs.get_support()) if s]
    cv = cross_val_score(make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5)),
                         Xv_train[cols], yv_train, cv=5).mean()
    scores[n] = (cv, cols)

plt.plot(list(scores), [v[0] for v in scores.values()], marker='o')
plt.title('Number of Features vs CV Accuracy'); plt.xlabel('Number of features'); plt.ylabel('CV accuracy')
plt.grid(True); plt.show()

best_n = max(scores, key=lambda n: scores[n][0])
best_features = scores[best_n][1]
print("Best number of features:", best_n)
print("Best CV accuracy:", scores[best_n][0])
print("Selected features:", best_features)

In [ ]:
ks = range(1, 31)
cv_acc, test_acc = [], []
for k in ks:
    pipe = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=k))
    cv_acc.append(cross_val_score(pipe, Xv_train[best_features], yv_train, cv=5).mean())
    pipe.fit(Xv_train[best_features], yv_train)
    test_acc.append(pipe.score(Xv_test[best_features], yv_test))

plt.plot(ks, cv_acc, marker='o', label='CV accuracy (train)')
plt.plot(ks, test_acc, marker='s', label='Test accuracy')
plt.title('K Value vs Accuracy (selected features)')
plt.xlabel('K'); plt.ylabel('Accuracy'); plt.legend(); plt.grid(True); plt.show()

best_k = list(ks)[int(np.argmax(cv_acc))]
print("Best k (by CV):", best_k, "| CV acc:", max(cv_acc))

final = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=best_k)).fit(Xv_train[best_features], yv_train)
yf = final.predict(Xv_test[best_features])
print("Final test accuracy:", accuracy_score(yv_test, yf))
print(confusion_matrix(yv_test, yf))
print(classification_report(yv_test, yf, target_names=['female', 'male']))

'k' is chosen by the highest 5-fold cross-validation accuracy on the training set. Very small 'k' overfits to noise, very large 'k' underfits. The chosen 'k' is the point where CV accuracy peaks/plateaus.